# MGKT Mini Explorer — Exploratory Analysis
Figures are saved to `reports/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path('..').resolve()))
from src.load_data import load_clean_data

REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

df = load_clean_data('../data/raw/data.csv')

In [ ]:
# Compute total knowledge score: sum of all 32 QXS columns (right - wrong per question)
score_cols = [f'Q{i}S' for i in range(1, 33)]
df['total_score'] = df[score_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

## Figure 1 — Descriptive: Total Knowledge Score Distribution

In [ ]:
fig1, ax1 = plt.subplots(figsize=(9, 5))

ax1.hist(df['total_score'], bins=40, color='steelblue',
         edgecolor='white', linewidth=0.5)

med = df['total_score'].median()
ax1.axvline(med, color='#E05C5C', linestyle='--', linewidth=1.5,
            label=f'Median = {med:.0f}')

ax1.set_xlabel('Total Score (sum of correct − incorrect across 32 questions)', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title(
    f'MGKT — Distribution of Total Knowledge Score  (n = {len(df):,})',
    fontsize=12, fontweight='bold'
)
ax1.legend(fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

out1 = REPORTS / 'fig1_score_distribution.png'
fig1.savefig(out1, dpi=150)
plt.show()
print(f'Saved: {out1.resolve()}')

## Figure 2 — Relational: Total Score by Gender

In [ ]:
gender_map = {1: 'Male', 2: 'Female', 3: 'Other'}
df['gender_label'] = df['gender'].map(gender_map)

gender_order = ['Male', 'Female', 'Other']
colors = ['#4C9BE8', '#E87C6A', '#7DC98F']

groups = [df.loc[df['gender_label'] == g, 'total_score'].dropna()
          for g in gender_order]

fig2, ax2 = plt.subplots(figsize=(8, 5))

bp = ax2.boxplot(
    groups,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    widths=0.5
)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Annotate n and median per group
for i, (g, grp) in enumerate(zip(gender_order, groups), start=1):
    ax2.text(i, ax2.get_ylim()[0] - 0.5,
             f'n={len(grp):,}\nmed={grp.median():.0f}',
             ha='center', va='top', fontsize=8.5)

ax2.set_xticks([1, 2, 3])
ax2.set_xticklabels(gender_order, fontsize=11)
ax2.set_xlabel('Gender', fontsize=11)
ax2.set_ylabel('Total Score', fontsize=11)
ax2.set_title(
    'MGKT — Total Knowledge Score by Gender\n'
    '(Males score slightly higher on median; all groups show wide spread)',
    fontsize=12, fontweight='bold'
)
ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

out2 = REPORTS / 'fig2_score_by_gender.png'
fig2.savefig(out2, dpi=150)
plt.show()
print(f'Saved: {out2.resolve()}')